# FastAPI 기반 AI 모델 서빙 교안

## 1. 배경 및 개요

### 1.1 모델 서빙과 FastAPI

AI 모델은 **훈련(Training)** 단계를 통해 데이터를 학습합니다.  
하지만 학습된 모델은 그 상태로는 **연구자가 직접 코드를 실행해야만** 결과를 볼 수 있습니다.

예를 들어 "고양이 vs 강아지 분류 모델"을 학습했다면:
- 연구자는 `predict("dog.jpg")` 같은 코드를 실행해서 결과를 볼 수 있습니다.
- 하지만 일반 사용자는 Python을 모를 수도 있고, 코드를 실행할 수도 없습니다.

그래서 필요한 것이 **모델 서빙(Model Serving)** 입니다.  
→ 학습된 모델을 **웹 서비스**처럼 배포해서, 누구나 **요청(request)을 보내면 결과(response)를 받을 수 있게 만드는 과정**입니다.  
→ 서빙을 하면 웹사이트, 모바일 앱, 다른 서버 어디에서든 모델을 사용할 수 있습니다.

**FastAPI**는 **Python 함수를 웹 API로 만들어 주는** 웹 프레임워크입니다.  
간단한 계산, DB 조회 같은 일반 기능은 물론, AI 모델 추론도 함수 안에 넣기만 하면 그대로 API가 됩니다.  
즉, **모델 서빙 = 모델 추론 함수를 FastAPI로 API로 만드는 것**입니다.

<img src="image/fastapi_overview.svg" width="800">

**그림 읽는 법**
- 클라이언트(웹, 앱, 다른 서버)가 HTTP 요청을 보내면 FastAPI가 ① 처리할 함수를 찾고 → ② 입력이 올바른지 검사하고 → ③ 함수를 실행해서 → ④ 결과를 JSON으로 돌려줍니다.
- ③에서 실행하는 함수는 계산이든 DB 조회든 무엇이든 될 수 있고, 이 강의에서는 여기에 **AI 모델 추론**을 넣습니다.
- 개발자는 함수와 입력 형식만 정의하면 되고, 주소 연결(①)·입력 검사(②)·JSON 변환(④)은 FastAPI가 자동으로 처리합니다.

### 1.2 왜 FastAPI인가?
- 비동기 처리 지원으로 DB 조회·외부 API 호출처럼 기다리는 작업이 많을 때도 여러 요청을 효율적으로 처리
- 자동 문서화(Swagger) 지원으로 API 테스트가 편리
- Python 친화적이며, 기존 AI 라이브러리(PyTorch, TensorFlow)와 호환 용이 


### 1.3 시장 수요
- AI 모델을 서비스에 연결하려는 연구자와 개발자 증가
- 모바일, 웹, 클라우드 환경에서 실시간 예측 필요
- 서버 개발 경험이 적은 초심자도 쉽게 배포 가능


### 1.4 학습 목표

- HTTP 기반 서버 구조, 요청/응답, 상태 코드 이해
- GET, POST 등 HTTP Method와 Path·Query·Body를 활용한 API 구현
- Pydantic을 활용한 요청·응답 데이터 검증
- JSON, 폼 데이터, 이미지 파일 등 다양한 입력 데이터 처리
- Swagger UI를 통해 테스트 및 디버깅 가능
- CORS의 개념을 이해하고 FastAPI에서 설정 가능
- FastAPI로 AI 모델 서빙 RESTful API 설계 및 구현 가능
- Amazon EC2, Google Cloud 등 클라우드 서버에 배포 가능

## 2. 모델 서빙 방법

AI 모델을 실제 서비스에서 사용하려면, 외부 프로그램이 모델을 호출할 수 있도록 **서빙(Serving)** 해야 한다.  
서빙 방식은 크게 **3가지 방법**으로 분류할 수 있다.

### 2.1 직접 API 서버 개발 (FastAPI, Flask)

- **방법:** Python 웹 프레임워크(FastAPI, Flask)를 사용해 직접 REST API 서버를 만든다.  
- **장점:** 필요한 기능을 자유롭게 커스터마이즈 가능. (전처리/후처리, 인증, 로깅 등)  
- **단점:** 서버 운영, 배포, 확장성, 트래픽 증가 대응 등을 모두 직접 관리해야 한다.

예시:  
사용자가 이미지를 업로드하면 `/predict`로 요청을 보내고,  
서버 내부에서 모델이 추론한 뒤 `"강아지"` 같은 JSON 결과를 반환한다.

<img src="image/fastapi_flow.png" width="600">

### 2.2 웹 기반 서빙(Web UI 방식)

- **방법:** 모델을 웹페이지 형태로 제공하여 사용자가 직접 브라우저에서 모델 기능을 사용할 수 있도록 제공한다.  
  (예: Gradio, Streamlit, HuggingFace Spaces)  
- **장점:** API나 서버 개발 없이도 빠르게 데모와 서비스 화면을 만들 수 있음.  
- **단점:** 고트래픽 서비스에는 부적합하며, UI 중심이라 복잡한 비즈니스 로직 구현은 제한적.

예시: 손글씨 숫자 인식 모델을 Gradio로 만들면, 사용자가 웹페이지에서 이미지를 올리면 바로 예측 결과가 표시됨.

<img src="image/wen.png" width="400">



### 2.3 서빙 전용 라이브러리 사용  
(TensorFlow Serving, TorchServe, Triton Inference Server)

- **방법:** 모델을 전용 서빙 엔진에 등록하면 자동으로 `/predict` API가 생성된다.  
- **장점:**  
  - 고성능 추론(배치 처리, GPU 최적화, 멀티 모델 배포)  
  - 표준화된 API 제공  
  - 운영 환경에 맞춘 최적화 기능 포함

- **단점(유연성이 떨어지는 이유):**  
  - **비즈니스 로직을 자유롭게 넣기 어려움**  
    - 예: 요청 포맷 커스터마이즈, 인증/로그인, 복잡한 후처리 등을 직접 넣기 어려움  
  - **내부 동작 방식이 고정되어 있어 구조 변경이 어려움**  
  - **‘모델 서빙’ 기능에만 초점**  
    → 서비스 로직을 모델 서버 안에 억지로 넣기 어려워 API 서버를 따로 만들어야 하는 경우가 많음  
  - **사용자 입력을 복잡하게 가공해야 하는 서비스에는 부적합**

예시:  
PyTorch 모델을 TorchServe에 등록하면 자동으로 `/predictions/model_name` API가 제공된다.

<img src="image/torchserve.png" width="400">

## 3. 백엔드 프로그래밍 기본

### 3.1 서버 구조 이해
서버(Server)는 **클라이언트(Client)의 요청(Request)을 받아 응답(Response)을 보내는 시스템**입니다.  
클라이언트는 브라우저, 스마트폰 앱, 다른 서버 등이 될 수 있습니다.

#### REST API란?

REST API(Representational State Transfer API)는 웹이나 앱이 서로 데이터를 주고받기 위한 **대표적인 통신 방식**입니다.  

웹에서 **자원(Resource)을 URL로 표현하고**, 그 자원을 **HTTP Method(GET/POST/PUT/DELETE 등)로 조작하는 방식**입니다.

자원(Resource)은 사용자 정보, 상품 정보, 게시글 등 **서버가 관리하는 데이터 또는 대상**을 의미합니다.

쉽게 말해:

- **URL = 어떤 자원인지 나타내는 주소** (예: `/users`, `/items/10`, `/predict`)
- **HTTP Method = 그 자원에 대해 어떤 행동을 할지 나타내는 규칙** (예: 조회, 생성, 수정, 삭제)

<img src="image/client_server_rest.svg" width="800">

REST API는 다음과 같은 원칙을 따릅니다:

1. **자원(Resource)은 URL(URI)로 표현한다.**  
   - `/users` → 사용자라는 자원  
   - `/users/1` → 아이디 1번 사용자

2. **행동(조작)은 HTTP Method로 표현한다.**  
   - `GET /users` → 사용자 목록 조회  
   - `POST /users` → 사용자 생성  
   - `PUT /users/1` → 사용자 정보 전체 수정  
   - `DELETE /users/1` → 사용자 삭제
   - URL에는 GET/POST 같은 단어가 표시되지 않으며, **HTTP Method는 요청의 첫 줄(요청 라인, Request Line)에 포함되어 구분된다.**

3. **Stateless(무상태성)**  
   - 이전 HTTP 요청에서 유지한 클라이언트의 상태에 의존하지 않고, 각 요청을 독립적으로 처리할 수 있어야 함
   - 매 요청이 독립적 → 확장성과 속도가 좋아짐

REST API는 현대 웹 서버와 모바일 앱 대부분이 사용하는 가장 표준적인 방식입니다.

### 3.2 HTTP Method (REST API에서 사용하는 행동 규칙)

- **GET**: 자원 조회(Read)  
  예: 뉴스 기사 목록 불러오기

- **POST**: 자원 생성(Create)  
  예: 회원가입 정보 저장

- **PUT**: 자원 전체 수정(Update – 전체 교체)  
  예: 프로필 전체 변경

- **PATCH**: 자원 부분 수정(Update – 일부 교체)  
  예: 닉네임만 변경

- **DELETE**: 자원 삭제(Delete)  
  예: 댓글 삭제

### 3.3 Request / Response 구조

클라이언트와 서버가 주고받는 메시지는 **요청(Request)** 과 **응답(Response)** 두 가지이고, 둘 다 같은 순서로 구성됩니다.

<img src="image/http_request_response.svg" width="800">

| 구성 | Request (요청) | Response (응답) |
|---|---|---|
| **Start Line** | 무엇을 요청하는지<br>Method + URL + HTTP 버전 | 결과가 어떤지<br>HTTP 버전 + 상태 코드 + 상태 메시지 |
| **Header** | 요청에 대한 부가 정보<br>예: `Content-Type`(데이터 형식), `Authorization`(인증 토큰) | 응답에 대한 부가 정보<br>예: `Content-Type`, `Content-Length`(데이터 크기) |
| **Body** | 서버로 보내는 실제 데이터 (선택)<br>예: JSON, 업로드한 이미지 | 클라이언트에게 보내는 실제 데이터<br>예: 예측 결과(JSON), HTML 페이지 |

### 3.4 HTTP Status Code (예시 포함)

- **200 OK**  
  - 의미: 요청이 정상적으로 처리됨  
  - 예시:  ```GET /users/1 → 사용자의 정보가 정상적으로 응답됨```

- **400 Bad Request**  
  - 의미: 요청 형식이 잘못됨(필수 데이터 누락, JSON 오류 등)  
  - 예시:  
    ```
    POST /users  (body에 name이 없을 때)
    {"age": 20 }
    → 서버: "name 필드가 없습니다" → 400 오류
    ```

- **401 Unauthorized**  
  - 의미: 인증 토큰 없음 또는 만료  
  - 예시:  
    ```
    GET /mypage  (Authorization 토큰 없음)
    → 로그인 필요 → 401 발생
    ```

- **404 Not Found**  
  - 의미: 요청한 자원이 존재하지 않음  
  - 예시:  
    ```
    GET /users/99999
    → 99999번 사용자 없음 → 404 반환
    ```

- **422 Unprocessable Entity**
  - 의미: 요청 형식은 맞지만 데이터 값이 규칙에 맞지 않음 (FastAPI가 입력 검증에 실패하면 반환)
  - 예시:
    ```
    GET /users/abc  (user_id는 정수여야 함)
    → 422 반환
    ```

- **500 Internal Server Error**  
  - 의미: 서버 내부에서 예기치 않은 오류 발생  
  - 예시:  
    ```
    GET /products
    → DB 연결 오류 발생 → 서버가 500 반환
    ```

### 3.5 요청과 응답 예시 및 해석

**요청(Request)**  
```
POST /predict HTTP/1.1
Host: example.com
Content-Type: application/json
Content-Length: 48

{
  "image_url": "https://example.com/cat.jpg"
}
```

**응답(Response)**  
```
HTTP/1.1 200 OK
Content-Type: application/json

{
  "result": "cat",
  "confidence": 0.98
}
```

- 클라이언트가 example.com 서버의 `/predict` 경로로 POST 요청을 보냄  
- 요청 데이터는 JSON 형식이며, Body에 이미지 URL이 포함되어 있음  
- Content-Length는 Body 데이터의 바이트 길이를 의미함  

- 서버는 요청을 정상적으로 처리하여 200 OK를 반환함  
- 응답 데이터는 JSON 형식이며, 예측 결과(cat)와 신뢰도(0.98)를 포함함

### 3.6 IP와 Port 개념

- **IP 주소**
  - 네트워크에서 컴퓨터(또는 스마트폰, 서버)가 어디에 있는지 구분하는 “주소” 역할을 합니다.
  - 이 주소 덕분에 우리가 보내는 요청이 정확한 장치로 전달될 수 있습니다.
  - IP는 두 종류로 나뉩니다.
    - **공인 IP**: 인터넷 전체에서 통하는 주소 (예: `203.0.113.15`)
    - **사설 IP**: 집·회사처럼 하나의 네트워크 안에서만 통하는 주소. 공유기가 각 장치에 나눠 줍니다. (예: `192.168.0.10`)

- **Port 번호**
  - 한 컴퓨터(IP) 안에서도 여러 개의 “방(서비스)”이 동시에 실행될 수 있습니다.
  - Port는 그 방의 번호로, 어떤 서비스로 들어갈지 알려주는 역할을 합니다.
  - 예:
    - 80번 → 일반 웹사이트(HTTP)
    - 443번 → 보안 연결 웹사이트(HTTPS)
    - 3306번 → MySQL DB
    - 6379번 → Redis

- **비유**
  - **공인 IP = 아파트 단지 주소** (외부에서 찾아오는 주소)
  - **사설 IP = 단지 안의 특정 호수** (단지 안에서 어느 집인지)
  - **Port = 그 집 안의 방 번호** (집 안에서 어느 방인지)
  - 예를 들어,
    - “서울시 강남구 OO아파트(=공인 IP)”의
    - “101동 1502호(=사설 IP)” 안에 있는
    - “3번 방(=Port)”을 찾아가는 것이라고 생각하면 됩니다.

- **간단한 실제 예시**
  - 아래 주소로 접속하면…
    ```
    http://192.168.0.10:8000
    ```
    - IP = 192.168.0.10 (사설 IP → 같은 네트워크 안의 컴퓨터)
    - Port = 8000 (FastAPI 개발 서버에서 자주 사용)
  - → 즉, “192.168.0.10 컴퓨터의 8000번 방(서비스)에 접속하겠다”는 뜻입니다.

### 3.7 실습: 크롬 개발자 도구로 진짜 요청·응답 보기

지금까지 배운 요청과 응답은 이론 속 이야기가 아닙니다.  
웹사이트를 열거나 검색창에 글자를 칠 때마다, 브라우저는 실제로 HTTP 요청을 보내고 응답을 받고 있습니다.

크롬의 **개발자 도구(F12)** 를 열면 이 요청·응답을 직접 볼 수 있습니다.

#### 준비
1. 크롬에서 새 탭을 엽니다.
2. `F12`를 누릅니다. (Mac: `Cmd + Option + I`)
3. 위쪽 탭 중 **Network(네트워크)** 를 클릭합니다.
4. 개발자 도구를 연 상태에서 주소창에 주소를 입력합니다. (개발자 도구가 열려 있어야 요청이 기록됩니다)

- 아래 실습의 `jsonplaceholder.typicode.com`은 연습용으로 공개된 무료 API 서버입니다.
- 크롬 언어가 한국어라면 메뉴도 한국어(헤더, 페이로드, 미리보기, 응답 등)로 보입니다.

#### 실습 ① GET 요청 보기 — `GET /users/1`

주소창에 입력:
```
https://jsonplaceholder.typicode.com/users/1
```

Network 목록에 생긴 `1`을 클릭하면, 3.3 그림에서 본 요청·응답의 각 부분을 그대로 찾을 수 있습니다.

| 개발자 도구에서 볼 곳 | 보이는 값 | 배운 개념 |
|---|---|---|
| Headers → General → Request URL | `https://jsonplaceholder.typicode.com/users/1` | URL = 자원 (3.1) |
| Headers → General → Request Method | `GET` | HTTP Method (3.2) |
| Headers → General → Status Code | `200` | 상태 코드 (3.4) |
| Headers → General → Remote Address | `서버IP:443` | IP와 Port (3.6) |
| Headers → Response Headers → content-type | `application/json; charset=utf-8` | Header (3.3) |
| Preview / Response 탭 | `{"id": 1, "name": "Leanne Graham", ...}` | Body (3.3) |

Remote Address 끝의 `443`은 3.6에서 본 HTTPS 포트 번호입니다.

#### 실습 ② 404 에러 보기 — `GET /users/99999`

```
https://jsonplaceholder.typicode.com/users/99999
```

- Status Code가 **404** 로 바뀝니다. → 3.4의 "99999번 사용자 없음 → 404" 예시와 같은 상황
- Response 탭의 Body는 빈 JSON `{}` 입니다.

#### 실습 ③ POST 요청 직접 보내기 — `POST /users`

주소창으로는 GET 요청만 보낼 수 있으므로, POST는 **Console(콘솔)** 탭에서 JavaScript로 보냅니다.

1. jsonplaceholder 페이지가 열린 탭에서 **Console** 탭을 클릭합니다.
2. 아래 코드를 붙여넣고 Enter를 누릅니다. (붙여넣기 경고가 뜨면 안내대로 `allow pasting`을 입력한 뒤 다시 붙여넣기)

```javascript
fetch("https://jsonplaceholder.typicode.com/users", {  // 요청을 보낼 주소
  method: "POST",                                      // Start Line의 Method
  headers: { "Content-Type": "application/json" },     // Header: Body가 JSON이라고 알림
  body: JSON.stringify({ name: "minsu", age: 25 })      // Body: 서버로 보낼 데이터
})
  .then(res => res.json())                             // 응답 Body를 JSON으로 읽기
  .then(data => console.log(data));                    // 응답 내용 출력
```

3. **Network** 탭으로 돌아가 새로 생긴 `users` 항목을 클릭합니다.

| 개발자 도구에서 볼 곳 | 보이는 값 | 의미 |
|---|---|---|
| Headers → Request Method | `POST` | 생성 요청 |
| Headers → Status Code | `201` | 201 Created = 새 자원이 생성됨 |
| **Payload** 탭 | `{"name": "minsu", "age": 25}` | 우리가 보낸 요청 Body |
| Response 탭 | `{"name": "minsu", "age": 25, "id": 11}` | 서버가 돌려준 응답 Body |

#### 실습 ④ 일상 속 요청 찾아보기 — 검색어 자동완성

1. Network 탭의 필터에서 **Fetch/XHR** 을 선택합니다. (페이지가 화면 뒤에서 보내는 데이터 요청만 보기)
2. 네이버나 구글 검색창에 글자를 하나씩 입력합니다.
3. 글자를 칠 때마다 요청이 하나씩 생깁니다. → 자동완성 목록은 이렇게 매번 서버에 물어봐서 받아오는 것입니다.
4. 요청 URL을 보면 `...?q=입력한글자` 형태입니다. → `?` 뒤의 값은 4.5에서 배울 **Query Parameter** 입니다.

필터를 다시 **All** 로 바꾸고 새로고침해 보면, 웹페이지 하나를 여는 데에도 HTML, 이미지, CSS 등 수십 개의 요청이 오가는 것을 볼 수 있습니다.

### 3.8 동기 / 비동기 처리

#### 동기(Synchronous)
- 요청을 **하나씩 순서대로 처리**하는 방식  
- 앞선 요청이 끝나야만 다음 요청이 실행됨  
- 단순하지만, 여러 요청이 몰리면 기다림이 길어질 수 있음

- **쉬운 비유**  
  - ATM 기기 1대 앞에서 줄을 서는 상황  
  - 앞사람이 뭔가를 오래 하면 내가 계속 기다려야 함

- **실제 예시**  
  - 서버가 이미지 1장을 처리하는 동안 다음 요청이 대기  
  - 파일 업로드를 동기 처리하면, 파일 하나 전송 끝날 때까지 다음 업로드가 진행되지 않음

#### 비동기(Asynchronous)
- 어떤 작업이 **응답을 기다리는 동안, 그 시간에 다른 요청을 먼저 처리**하는 방식
- 기다리는 시간이 긴 작업(DB 조회, 외부 API 호출, 파일 전송 등)이 많을수록 전체 효율이 높아짐
- 실시간 서비스에서 매우 유용 (채팅, 알림 등)

- **쉬운 비유**
  - 패스트푸드점 주문 시스템
  - 주문만 받고 나중에 번호를 불러주는 방식
  - 손님은 기다리는 동안 다른 일을 할 수 있고, 주방은 동시에 여러 주문을 처리함

- **실제 예시**
  - 서버가 외부 번역 API의 응답을 기다리는 동안 다른 사용자의 요청을 처리
  - 웹사이트가 여러 사용자의 채팅 메시지를 실시간으로 처리
  - 비동기 방식의 대표 기술: Node.js, Python의 `async/await`

> 비동기가 아껴주는 것은 **"기다리는 시간"** 입니다.  
> 모델 추론처럼 CPU·GPU가 계속 계산하는 작업은 비동기로 바꿔도 빨라지지 않습니다. (8장에서 다시 다룹니다)

## 4. FastAPI 소개 및 기본 지식

### 4.1 FastAPI 특징
**FastAPI**는 최근 가장 많이 사용되는 Python 웹 프레임워크 중 하나입니다.  

<img src="image/swagger.png" width="700">

- **Python 기반, 고성능**  
  → 비동기(Async) 처리와 함께 매우 빠른 속도를 제공.  
- **자동 문서화 지원**  
  → 코드를 작성하면 Swagger UI, ReDoc을 통해 API 문서를 자동으로 확인 가능.  
- **비동기 처리 지원**  
  → DB 조회, 외부 API 호출처럼 기다리는 작업이 많을 때 여러 요청을 효율적으로 처리할 수 있음. 

### 4.2 FastAPI의 기본 문법: 데코레이터, 타입 힌트, 함수 구조

#### ✔ 데코레이터(Decorator)란?
Python에서 함수 위에 `@`를 붙여 함수의 동작을 수정하거나 확장하는 문법입니다.

**기본 형태:**
```python
@app.get("/경로")  # ← 데코레이터
def 함수명(매개변수):
    return 데이터
```

**의미:**
- `@app.get()` = "이 함수를 GET 요청이 들어올 때 실행해라"
- `/경로` = "어떤 URL에서 이 함수를 실행할지"
- `함수` = "실행할 실제 코드"

#### ✔ 실제 예시로 이해하기

```python 
from fastapi import FastAPI

app = FastAPI()

# 데코레이터: "이 함수는 GET /hello 요청을 처리해"
@app.get("/hello")
def hello():
    return {"message": "안녕하세요"}

# 클라이언트가 http://localhost:8000/hello 접속
# → FastAPI가 자동으로 hello() 함수 실행
# → {"message": "안녕하세요"} JSON 반환
```

#### ✔ 데코레이터 종류 (HTTP Method)

| 데코레이터 | 역할 | 예시 |
|-----------|------|------|
| `@app.get()` | 조회(Read) | 사용자 정보 가져오기 |
| `@app.post()` | 생성(Create) | 새 데이터 저장 |
| `@app.put()` | 전체 교체(Update) | 기존 데이터 변경 |
| `@app.patch()` | 부분 수정(Update) | 닉네임만 변경 |
| `@app.delete()` | 삭제(Delete) | 데이터 제거 |

#### ✔ 왜 데코레이터를 사용하나?

**장점:**
- URL과 함수를 자동으로 연결 (라우팅)
- 코드가 간단하고 직관적
- FastAPI가 자동으로 문서화도 생성

#### 타입 힌트(Type Hints) - 함수 매개변수 정의

FastAPI의 강력한 기능은 **Python 타입 힌트**에서 나옵니다.

#### ✔ 타입 힌트란?
함수의 매개변수가 **어떤 데이터 타입인지 명시**하는 문법입니다.

```python
def read_item(item_id: int, q: str | None = None):  # item_id는 int, q는 str 또는 None
```

**각 부분 설명:**
- `item_id: int` = "item_id는 정수(int) 타입"
- `q: str | None` = "q는 문자열(str) 또는 None"
- `= None` = "기본값은 None (없어도 OK)"

#### ✔ 타입 힌트의 효과

**1) 자동 타입 검증**
```python
# 잘못된 요청
GET /items/abc?q=hello
     ↑ 문자열인데 정수 타입 요청
→ FastAPI: "item_id는 정수여야 합니다" (422 오류)
```

**2) 자동 문서 생성**
- Swagger UI에서 "item_id는 정수" 자동 표시
- 사용자가 올바른 형식으로 입력하도록 유도

**3) IDE 자동완성**
- 개발 도구가 타입을 인식해 코드 자동완성 제공

#### ✔ 자주 사용하는 타입들

```python
# 기본 타입
item_id: int           # 정수
price: float           # 실수
name: str              # 문자열
is_active: bool        # 참/거짓

# 선택적 타입 (Python 3.10+)
q: str | None = None   # 문자열 또는 None

# 구 버전 Python
from typing import Optional
q: Optional[str] = None  # 위와 동일
```

#### ✔ 실제 예시

```python
@app.get("/users/{user_id}")
def get_user(user_id: int, limit: int = 10):
    # user_id: 정수 (필수)
    # limit: 정수 (선택, 기본값 10)
    return {"user_id": user_id, "limit": limit}
```

##### 올바른 요청
GET /users/123?limit=20
→ {"user_id": 123, "limit": 20}

##### 잘못된 요청
GET /users/abc
→ 422 오류: user_id는 정수여야 합니다
```

#### 함수의 반환값과 JSON

FastAPI는 우리가 반환하는 **Python 객체(딕셔너리, 리스트, Pydantic 모델 등)** 을  
클라이언트에게 전송할 때 **자동으로 JSON 형식으로 변환하여 응답**합니다.  

즉, 개발자는 JSON 문자열을 직접 만들 필요가 없고,  
단순히 Python 객체를 `return`하기만 하면 됩니다.

```python
@app.get("/items/{item_id}")
def read_item(item_id: int, q: str | None = None):
    # 여기서 item_id와 q는 URL로 전달된 값이며, JSON이 아님
    return {
        "received_id": item_id,
        "search_keyword": q,
        "message": "요청을 잘 받았습니다!"
    }

# 위의 Python 딕셔너리를 FastAPI가 자동으로 JSON으로 변환하고
# Content-Type, 상태 코드 등의 HTTP 응답을 생성하여 클라이언트에 전송함
```

**이 “자동 JSON 변환”이 FastAPI의 대표적인 장점입니다!**

### 4.3 설치 및 기본 구조
FastAPI 설치와 실행은 매우 간단합니다.

#### 1) FastAPI 설치
```bash
pip install "fastapi[standard]"
```
`fastapi[standard]`는 FastAPI 본체와 함께 자주 쓰는 도구를 한 번에 설치합니다.
- `uvicorn` : FastAPI 앱을 실행하는 웹 서버
- `python-multipart` : 폼 데이터·파일 업로드를 받을 때 필요한 라이브러리

#### 2) 기본 앱 파일(main.py) 생성
아래 셀을 실행하면 현재 폴더에 `main.py` 파일이 만들어집니다. (`%%writefile 파일명` = 셀 내용을 파일로 저장)  
직접 `main.py` 파일을 만들고 코드를 붙여넣어도 됩니다.

In [ ]:
%%writefile main.py
from fastapi import FastAPI                         # FastAPI 클래스 불러오기

app = FastAPI()                                     # 앱 객체 생성 (uvicorn이 실행할 대상)

@app.get("/")                                       # GET / 요청이 오면 아래 함수 실행
def root():
    return {"message": "Hello, FastAPI"}            # dict를 반환하면 JSON으로 자동 변환

@app.get("/items/{item_id}")                        # {item_id} 자리의 값이 Path Parameter로 전달됨
def read_item(item_id: int, q: str | None = None):  # item_id는 필수 정수, q는 선택 문자열(Query)
    return {"item_id": item_id, "q": q}             # 받은 값을 그대로 돌려줌

#### 3) 개발 서버 실행
**터미널**에서 `main.py`가 있는 폴더로 이동한 뒤 실행합니다.

```bash
uvicorn main:app --reload
```
- `main:app` → `main.py` 파일 안의 `app` 객체를 실행
- `--reload` 옵션 → 코드를 수정하면 서버가 자동으로 재시작되어 개발할 때 매우 편리합니다.

### 4.4 FastAPI 서버 테스트 방법

FastAPI는 테스트하는 방법이 다양합니다.  
초심자도 바로 따라 할 수 있도록 3가지 방식을 소개합니다.

#### 방법 1) 웹 브라우저 테스트 (가장 간단)

아래 주소를 브라우저에 입력:

```http://127.0.0.1:8000/```

→ JSON 응답:  ```{"message": "Hello, FastAPI"}```


특정 아이템 조회: ```http://127.0.0.1:8000/items/123?q=hello```

→ JSON 응답:  ```{"item_id":123,"q":"hello"}```

#### 방법 2) curl로 터미널에서 테스트

터미널에서 `curl` 명령을 실행합니다. 노트북 셀에서는 앞에 `!`를 붙이면 됩니다.  
(`-s` : 진행 표시 없이 결과만 출력)

In [ ]:
!curl -s http://127.0.0.1:8000/                          # 루트 경로에 GET 요청

아이템 조회 (Path Parameter `999`, Query Parameter `q=python`):

In [ ]:
!curl -s "http://127.0.0.1:8000/items/999?q=python"      # ?가 들어간 주소는 따옴표로 감싸기

#### 방법 3) 자동 문서화된 Swagger UI 사용

FastAPI는 자동으로 API 문서를 생성합니다.

브라우저에서 아래 주소 접속:

```
http://127.0.0.1:8000/docs
```

→ GET/POST 버튼을 눌러 API를 직접 실행해볼 수 있음  
→ Request/Response 구조를 자동으로 보여줘서 매우 편리  
→ 초보자/테스터/기획자도 쉽게 사용 가능

ReDoc 스타일 문서도 제공됨:

```
http://127.0.0.1:8000/redoc
```

#### 방법 4) 크롬 개발자 도구로 확인

3.7에서 쓴 개발자 도구로 내 FastAPI 서버의 요청·응답도 볼 수 있습니다.

1. `F12` → Network 탭을 연 상태에서 `http://127.0.0.1:8000/items/123?q=hello` 접속 → Status `200`, Response 탭에 JSON
2. `http://127.0.0.1:8000/items/abc` 접속 → Status `422` (item_id가 정수가 아니므로 FastAPI가 거절)
3. Swagger UI(`/docs`)에서 Execute를 누를 때도 Network 탭에 요청이 기록됩니다.
4. 기록된 요청을 우클릭 → **Copy → Copy as cURL** 을 누르면, 같은 요청을 보내는 curl 명령어가 복사됩니다. (방법 2와 연결)

#### 요약

| 테스트 방식 | 특징 |
|------------|------------------------------------------------|
| **웹 브라우저** | GET 요청 빠른 확인 |
| **curl** | 터미널에서 간단 자동화 테스트 |
| **Swagger UI (/docs)** | 가장 편리 • 요청/응답 확인 • 파라미터 테스트 가능 |
| **개발자 도구 (F12)** | 실제로 오간 요청·응답의 Header, Body, 상태 코드 확인 |

### 실습 문제 1. 현재 시간을 알려주는 API 만들기 — `/time`

지금까지 배운 내용을 토대로, **아주 간단한 API 하나를 main.py에 직접 추가해보세요.**

**API 기능 요구 사항**
1. 아래 URL로 요청을 보내면 서버가 아래처럼 응답하도록 만들어보세요
 - 요청 : `GET` http://127.0.0.1:8000/time (브라우저 주소창에 입력)
 - 응답 : {"message": "현재 시간은 2025년 1월 1일 15시 30분 입니다."}
2. **응답 형식(JSON)** : `{"message": "현재 시간은 X년 X월 X일 X시 X분 입니다."}`
3. 현재 시간을 **한국 시간(KST)** 기준으로 표현

힌트: 아래 함수는 “현재 시간을 문자열로 반환하는 함수”입니다.
**이 함수를 main.py에 그대로 붙여넣어도 됩니다.**

In [ ]:
from datetime import datetime, timedelta, timezone  # 날짜·시간과 시간대를 다루는 표준 라이브러리

def get_korean_time_string():
    KST = timezone(timedelta(hours=9))              # 한국 시간대(KST) = UTC+9시간
    now = datetime.now(KST)                         # 한국 시간 기준 현재 시각

    return f"현재 시간은 {now.year}년 {now.month}월 {now.day}일 {now.hour}시 {now.minute}분 입니다."  # 문장으로 조립해 반환

#### ✔ 테스트 방법

- 브라우저로 테스트  
  ```
  http://127.0.0.1:8000/time
  ```
- Swagger UI로 테스트  
  ```
  http://127.0.0.1:8000/docs
  ```
  → GET /time 클릭 → Execute

- 터미널로 테스트
  ```
  curl http://localhost:8000/time
  ```

<details>
<summary>정답 코드 펼치기</summary>

```python
# main.py에 이어서 추가 (app = FastAPI()는 이미 있음)

from datetime import datetime, timedelta, timezone  # 시간 계산용

@app.get("/time")                                   # GET /time 요청 처리
def get_korean_time_string():
    KST = timezone(timedelta(hours=9))              # 한국 시간대(UTC+9)
    now = datetime.now(KST)                         # 한국 시간 기준 현재 시각
    message = f"현재 시간은 {now.year}년 {now.month}월 {now.day}일 {now.hour}시 {now.minute}분 입니다."  # 응답 문장 만들기
    return {"message": message}                     # {"message": ...} JSON으로 응답
```
</details>

### 4.5 Path & Query Parameter

이제 FastAPI에서 **실제로 데이터를 받고 보내는 방법**을 배워보겠습니다.  
FastAPI에서 API로 값을 전달하는 주요 방식은 두 가지입니다:

#### ✔ 1) Path Parameter (필수 정보)
URL 경로의 일부가 되어, **어떤 자원(resource)을 요청하는지 지정하는 값**입니다.

예:
```
/items/10
→ item_id = 10번 아이템을 조회하겠다
```

Path Parameter는 **반드시 있어야 하는 값**이며,  
보통 “특정한 하나의 대상”을 찾을 때 사용합니다.  

비유: **아파트 주소(건물동/호수)를 찾는 것 → 꼭 있어야 하는 정보**


#### ✔ 2) Query Parameter (선택 정보)
`?` 뒤에 붙는 값으로, **검색 조건·필터·옵션 같은 추가 정보**입니다.

예:
```
/items/10?q=apple
→ item_id=10번 아이템 중에서 q=apple 조건을 적용
```

Query Parameter는 **선택적(optional)** 이며,  
주로 “추가 조건” 또는 “옵션”을 제공할 때 사용합니다.

비유:
- **택배 요청 시 선택하는 옵션 (문 앞 배송, 경비실 보관 등)**  
- 주소는 고정이지만 옵션은 선택하는 것


#### ✔ 3) 실제 FastAPI 코드 예시

```python
from fastapi import FastAPI

app = FastAPI()

@app.get("/items/{item_id}")                        # 경로의 {item_id} → Path Parameter
def read_item(item_id: int, q: str | None = None):  # 경로에 없는 매개변수 q → Query Parameter
    return {"item_id": item_id, "q": q}
```

- `item_id` → **Path Parameter** (필수)
- `q` → **Query Parameter** (선택)
  - `None` = 기본값 None (= 전달 안 해도 OK)

#### 요약

| 구분 | Path Parameter | Query Parameter |
|------|----------------|-----------------|
| 목적 | 어떤 자원을 요청하는지 지정 | 옵션·필터·검색 조건 |
| 필수 여부 | 필수 | 선택 |
| URL 형태 | `/items/10` | `/items/10?q=apple` |
| 비유 | 아파트 주소 | 배달 옵션 |


이렇게 두 파라미터의 **역할**과 **언제 사용하는지**가 정확히 구분되면 헷갈리지 않습니다.

### 문제 1. FastAPI에서 Path Parameter와 Query Parameter의 정의와 예를 하나씩 쓰시오.  

<details>
<summary>정답 보기</summary>

Path Parameter 예시
- 정의: URL 경로의 일부  
- 형식: `/items/{item_id}`  
- 요청 예: `/items/10`  
→ Path Parameter = `10`

Query Parameter 예시
- 정의: `?` 뒤에 오는 추가 옵션  
- 형식: `?q=검색어`  
- 요청 예: `?q=apple`  
→ Query Parameter = `q=apple`

</details>


### 문제 2. FastAPI에서 Path Parameter와 Query Parameter의 차이를 간단히 설명하시오.

<details>
<summary>정답 보기</summary>

- **Path Parameter:**  
  URL 경로 자체에 포함되어 “특정 자원”을 지정하는 필수 정보  
  예: `/users/1`

- **Query Parameter:**  
  옵션·필터·검색 조건 등 “추가 정보”를 전달하는 선택적 값  
  예: `/users/1?active=true`

</details>

### 4.6 POST 방식 & Request Body 기초 이해하기

**POST 방식과 Request Body(JSON)** 를 다뤄봅니다.  
지금까지는 URL로 전달되는 *Path*와 *Query* 파라미터만 사용했지만,  
서버에 **JSON 데이터를 보내서 처리하려면 POST + Request Body**를 사용해야 합니다.

#### ✔ POST는 왜 필요한가?
- GET은 “데이터 주세요” → Body를 보내지 않음
- POST는 “데이터 보낼 테니 처리해주세요” → Body(JSON)를 보낼 수 있음

즉, **서버로 JSON 데이터를 보낼 때는 POST 방식이 자연스럽고 표준적**입니다.

#### ✔ Request Body(JSON)란?
클라이언트가 서버로 보내는 실제 데이터입니다. 아래처럼 생긴 JSON을 서버로 전달합니다.

```json
{
  "name": "홍길동",
  "hobby": "독서"
}
```

FastAPI에서는 Request Body를 가장 간단히 다음처럼 받을 수 있습니다:

```python
@app.post("/introduce")                             # POST /introduce 요청 처리
def introduce(body: dict):                          # JSON Body가 dict로 변환되어 body에 들어옴
    name = body.get("name")                         # dict에서 "name" 값 꺼내기
    hobby = body.get("hobby")                       # dict에서 "hobby" 값 꺼내기
    return {"message": f"{name}님의 취미는 {hobby}입니다."}  # dict 반환 → JSON 응답
```

응답:
```json
{"message": "홍길동님의 취미는 독서입니다."}
```

즉,
- `body: dict` : JSON이 자동으로 파이썬 딕셔너리로 변환되어 들어옴
- `body.get("키")` : dict에서 원하는 값 꺼내기

이 두 가지만 이해하면 **Request Body 기초는 완전히 끝!**

#### ✔ FastAPI는 값을 어디서 꺼낼지 어떻게 알까?
FastAPI는 HTTP Method(GET/POST)가 아니라 **매개변수의 타입**을 보고 판단합니다.

| 매개변수 | FastAPI의 해석 |
|---|---|
| 경로의 `{이름}`과 같은 이름 | Path Parameter |
| `int`, `str`, `bool` 같은 단순 타입 | Query Parameter |
| `dict` 또는 Pydantic 모델 | Request Body |

그래서 Body로 받을 값은 `name: str, age: int`처럼 따로 적지 않고, `body: dict`처럼 **하나의 묶음**으로 받아야 합니다.

### 실습 문제 2. JSON Body를 받아서 그대로 응답하는 `/echo` API 만들기

아래 조건을 만족하는 새로운 API를 **main.py에 직접 추가**하세요.

#### 요구사항
1) `POST /echo` 엔드포인트 만들기  
2) JSON Body에 `"message"` 값을 받기  
3) 서버는 `"received"` 키로 다시 돌려주기

#### 예시 입력(JSON Body)
```json
{
  "message": "hello"
}
```

#### 예시 출력(JSON Response)
```json
{
  "received": "hello"
}
```

힌트: 4.6에서 본 것처럼 JSON Body는 `body: dict` 매개변수로 받습니다.

Swagger UI(`http://127.0.0.1:8000/docs`)에서 테스트하면 가장 쉽습니다.

curl로 테스트하려면:
```bash
curl -X POST http://127.0.0.1:8000/echo \
  -H "Content-Type: application/json" \
  -d '{"message":"hello"}'
```

<details>
<summary>정답 코드 펼치기</summary>

```python
# main.py에 이어서 추가

@app.post("/echo")                                  # POST /echo 요청 처리
async def echo(body: dict):                         # JSON Body → dict
    message = body.get("message")                   # dict에서 "message" 값 꺼내기
    return {"received": message}                    # "received" 키로 돌려주기 (JSON 자동 변환)
```

</details>

### 실습 문제 3. 새로운 API 만들기: `/profile/{user_id}`

아래 조건을 만족하는 **새로운 API 하나를 직접 만들어보세요. (main.py에 추가)**  

**API 기능 요구 사항**
1. Request Body(JSON)를 함께 보내야 하므로 이 API는 **POST**로 구현합니다.
2. **Path Parameter**
- `user_id` (int)
- 예: `/profile/10` → user_id = 10

3. **Query Parameter**
- `verbose` (bool, 선택 / 기본값: false)
- 사용하는 방법:
  - `/profile/10?verbose=true`
  - 또는 생략하면 자동으로 `false` 처리됨  
  - (FastAPI는 `?verbose=true`, `?verbose=false` 를 자동으로 bool로 변환함)

4. **Request Body (JSON)**
아래와 같은 JSON 데이터를 Body로 전달해야 한다.

```json
{
  "name": "홍길동",
  "age": 30
}
```

> 힌트: Body는 4.6에서처럼 `body: dict` 매개변수로 받습니다.  
> `name: str, age: int`처럼 따로 적으면 FastAPI가 Query Parameter로 해석합니다.

5. **Response Body (JSON)**
서버는 다음과 같은 JSON 응답을 반환해야 한다.

```json
{
  "user_id": 10,
  "name": "홍길동",
  "age": 30,
  "verbose": true,
  "message": "Profile received"
}
```

<details> <summary> 테스트 방법 안내 (중요!)</summary>

아래는 다양한 테스트 방법을 제공하지만,    **초심자라면 Swagger UI만 사용해도 충분합니다.**  
curl 명령어는 선택 사항이며, 익숙하지 않아도 괜찮아요.

### 1) Swagger UI (가장 쉬운 방법 — 추천)
```
http://127.0.0.1:8000/docs
```
- `POST /profile/{user_id}` 클릭
- `user_id` 입력
- `verbose` 입력 (true / false 또는 비워두기 가능)
- JSON Body 입력 후 실행(Execute)

---

### 2) curl 테스트 (선택 사항)
초보자는 curl을 몰라도 전혀 문제 없습니다.  
단축 테스트를 원하면 아래 명령어를 실행해보세요.

```bash
curl -X POST "http://127.0.0.1:8000/profile/10?verbose=true" \
    -H "Content-Type: application/json" \
    -d "{\"name\": \"홍길동\", \"age\": 30}"
```

</details>

<details> <summary>정답 보기</summary>

```python
# main.py에 이어서 추가

@app.post("/profile/{user_id}")                     # {user_id} → Path Parameter
async def create_profile(user_id: int, body: dict, verbose: bool = False):  # body → Request Body, verbose → Query Parameter
    return {
        "user_id": user_id,                         # Path에서 받은 값
        "name": body.get("name"),                   # Body(JSON)에서 꺼낸 값
        "age": body.get("age"),
        "verbose": verbose,                         # Query에서 받은 값 (없으면 False)
        "message": "Profile received"
    }
```
</details>

## 5. Pydantic 활용

### 5.1 Pydantic이란?

**Pydantic**은 Python에서 데이터를 **검증(Validation)** 하고 **구조화(Modeling)** 하기 위한 라이브러리입니다.  
FastAPI는 Request Body를 받을 때 이 Pydantic을 활용하여:

- 데이터 유형이 맞는지 검사하고  
- 누락된 필드가 없는지 확인하고  
- 잘못된 입력이 있으면 자동으로 에러를 반환합니다.

#### ✔ 왜 굳이 Pydantic을 배우는가?

FastAPI는 **Pydantic 없이도 작동합니다.**  
예를 들어 다음처럼 `dict`로 요청을 받아도 됩니다:

```python
@app.post("/echo")
async def echo(body: dict):                         # dict로 받으면 내부 구조는 검사하지 않음
    return {"received": body.get("message")}
```

하지만 `dict` 방식은 **요청 데이터의 구조를 명확하게 검증하지 못한다**는 한계가 있습니다.

예를 들어, 아래처럼 중첩된 JSON이 들어온다고 가정해봅시다.

```json
{
  "user": {
    "age": "스물다섯"
  }
}
```

겉으로는 `dict` 형태이기 때문에 FastAPI는 일단 요청을 받을 수 있습니다.  
하지만 내부의 `user.age`가 숫자(int)여야 하는지, 문자열이면 안 되는지까지는 자동으로 검사하지 않습니다.  
즉, **dict는 “JSON처럼 생긴 데이터”는 받을 수 있어도, 그 안의 세부 구조와 타입까지 안전하게 검증해주지는 못합니다.**

그래서 API가 커지면 아래와 같은 문제가 생깁니다:

- 값이 빠져 있어도 오류가 바로 드러나지 않을 수 있다  
- 타입이 잘못 들어와도 내부에서 직접 확인해야 한다  
- 중첩된 JSON 구조의 각 필드까지 자동 검증되지 않는다  
- 어떤 구조의 JSON이 들어오는지 문서(스웨거)에 명확하게 표시되지 않는다  
- 잘못된 입력이 들어왔을 때 서버 로직 내부에서 뒤늦게 오류가 날 수 있다

이때 Pydantic을 사용하면 요청 데이터의 **필드, 타입, 중첩 구조, 기본값** 등을 미리 정의할 수 있고,  
잘못된 입력은 FastAPI가 자동으로 검증하여 오류를 반환합니다.

예를 들면 다음과 같습니다:

```python
from pydantic import BaseModel                      # Pydantic 모델의 기본 클래스

class UserInfo(BaseModel):                          # 안쪽 JSON {"age": ...} 의 구조
    age: int                                        # age는 반드시 정수

class UserAgeRequest(BaseModel):                    # 바깥 JSON {"user": {...}} 의 구조
    user: UserInfo                                  # user 안에 UserInfo 구조가 들어 있어야 함

@app.post("/user-age")
def user_age(body: UserAgeRequest):                 # 요청이 들어오면 구조·타입을 자동 검증
    return {"age": body.user.age}                   # 점(.)으로 중첩된 값에 접근
```

이제 아래와 같은 요청이 들어오면:

```json
{
  "user": {
    "age": "스물다섯"
  }
}
```

FastAPI는 이를 자동으로 검증하고, 잘못된 형식이므로 오류를 반환합니다.

그래서 실무에서는 **안전성 + 자동 문서화 + 유지보수성** 때문에 대부분 Pydantic을 사용합니다.

즉,  
> ✔ `dict`로도 동작은 하지만  
> ✔ **요청 데이터의 구조와 타입까지 안전하게 검증하려면 Pydantic이 필요하다**

### 5.2 Pydantic 기본 사용법 (FastAPI 연동)

#### 1) 먼저 데이터 모델(BaseModel) 정의

```python
from pydantic import BaseModel                      # Pydantic 모델의 기본 클래스

class User(BaseModel):                              # BaseModel을 상속해 데이터 구조 정의
    username: str                                   # 필수, 문자열
    age: int                                        # 필수, 정수
    email: str | None = None                        # 선택, 없으면 None
```

- `BaseModel` 상속 → Pydantic 모델 정의
- 타입 힌트 기반으로 자동 검증
- `str | None = None` → 값이 없어도 허용 (기본값 None)

#### 2) FastAPI에서 Request Body에 적용

Pydantic 모델을 함수의 매개변수로 넣으면  
FastAPI가 자동으로 JSON을 파싱하고 검증합니다.

```python
@app.post("/users/")
async def create_user(user: User):                  # JSON Body → User 객체로 자동 변환·검증
    return {
        "username": user.username,                  # 속성(.)으로 값 꺼내기
        "age": user.age,
        "email": user.email
    }
```

#### ✔ 검증 실패 예시

요청:

```json
{
  "username": "홍길동",
  "age": "스물다섯"
}
```

FastAPI가 자동으로 **422** 상태 코드와 함께 아래와 같은 에러를 반환:

```json
{
  "detail": [
    {
      "type": "int_parsing",
      "loc": ["body", "age"],
      "msg": "Input should be a valid integer, unable to parse string as an integer",
      "input": "스물다섯"
    }
  ]
}
```

- `loc` : 문제가 생긴 위치 (Body의 `age` 필드)
- `msg` : 무엇이 잘못됐는지
- `input` : 실제로 들어온 값

### 5.3 폼 데이터 + 파일 업로드 처리

지금까지는 Body를 모두 **JSON**으로 보냈습니다. 그런데 사진 같은 **파일**을 보내야 할 때는 다른 방법이 필요합니다.

#### ✔ 폼(Form)이란?

**폼(Form)** 은 웹페이지에서 사용자가 값을 입력해서 서버로 보내는 **입력 양식**입니다. 이미 매일 쓰고 있습니다.

- 회원가입 화면: 이름·이메일 입력칸 + **가입하기** 버튼
- 게시판 글쓰기: 제목·내용 입력칸 + **사진 첨부** 버튼 + **등록** 버튼
- 프로필 사진 변경: **파일 선택** 버튼 + **저장** 버튼

버튼을 누르는 순간, 입력칸에 쓴 값과 첨부한 파일이 하나의 요청에 담겨 서버로 전송됩니다.  
이렇게 폼으로 보내는 데이터를 **폼 데이터(Form Data)** 라고 합니다.

#### ✔ 왜 JSON이 아니라 폼 데이터로 보낼까?

JSON은 **글자와 숫자만** 담을 수 있는 형식이라, 사진 같은 파일을 그대로 넣을 수 없습니다.

그래서 파일을 보낼 때는 Body를 **폼 데이터(`multipart/form-data`)** 라는 방식으로 포장합니다.

- 칸막이로 나뉜 택배 상자라고 생각하면 됩니다.
- 칸마다 값을 하나씩 넣습니다: 이름 칸, 나이 칸, 사진 파일 칸
- `multipart` = 여러(multi) 칸(part)으로 나뉘어 있다는 뜻입니다.

<img src="image/json_vs_form_data.svg" width="800">

| 포장 방식 | 담을 수 있는 것 | FastAPI에서 받는 법 |
|---|---|---|
| **JSON** | 글자·숫자 | `body: dict` 또는 Pydantic 모델 |
| **폼 데이터** | 글자·숫자 + 파일 | 글자 → `Form(...)`, 파일 → `UploadFile = File(...)` |

한 요청의 Body는 하나뿐이라 포장 방식도 하나만 고를 수 있습니다.  
그래서 파일이 포함된 요청에서는 이름·나이 같은 글자 값도 JSON이 아니라 **폼 데이터의 한 칸**으로 함께 보냅니다.

#### ✔ FastAPI 코드 예시

이름·나이(글자)와 파일 하나를 폼 데이터로 함께 받는 API입니다.

```python
from fastapi import FastAPI, File, Form, UploadFile # Form: 폼 데이터, File/UploadFile: 업로드 파일

app = FastAPI()

@app.post("/form-upload")
async def form_upload(
    name: str = Form(...),                          # 폼의 name 칸 (필수, 문자열)
    age: int = Form(...),                           # 폼의 age 칸 (필수, 정수로 자동 변환·검증)
    file: UploadFile = File(...)                    # 폼의 file 칸 = 업로드 파일 (필수)
):
    return {
        "name": name,                               # 폼에서 받은 값
        "age": age,
        "filename": file.filename                   # 업로드된 파일 이름
    }
```

- **`Form(...)`**: "이 값은 폼 데이터의 한 칸에서 꺼내라"는 표시
  - 타입 힌트대로 자동 검증·변환 (`age`에 "세살"을 보내면 422)
- **`UploadFile = File(...)`**: "이 칸에는 파일이 들어 있다"는 표시
  - `file.filename`으로 파일 이름 확인 (파일 내용을 읽어 모델에 넣는 방법은 8장에서 다룸)
  - 이미지, 문서, PDF 등 어떤 파일이든 받을 수 있음
- **`...`**: "필수 값"이라는 뜻 (안 보내면 422)

Swagger UI(`/docs`)에서 이 API를 열면 name, age 입력칸과 **파일 선택 버튼**이 자동으로 만들어집니다. 웹페이지의 폼과 똑같은 모양입니다.

#### 이 구조가 왜 중요한가?
- 실전 AI 서비스(예: 얼굴 인식, 제품 사진 분류 등)에서는
  **“파일(이미지)” + “요청 옵션(폼 데이터)”** 조합이 매우 흔하게 사용됨

#### 정리
| 구분 | 의미 | 예시 |
|------|------|------|
| **Request Body** | 서버로 보내는 실제 데이터 | JSON, 파일, 폼 데이터 |
| **Response Body** | 서버가 돌려주는 결과 | JSON, 메시지 |
| **주요 활용** | 모델 입력, 폼 제출, 파일 업로드 | 이미지 + 옵션(폼 데이터), 문서 + 메타데이터 |

### 5.4 정리

| 항목 | 설명 |
|------|------|
| 동작 자체는 dict도 가능함 | 하지만 검증이 없어서 위험 |
| Pydantic은 자동 검증 제공 | 잘못된 입력을 서버에 전달하지 않음 |
| Swagger 문서 자동 생성 | Request Body 구조가 정확히 표시됨 |
| 실무에서 거의 필수 | 안전성 + 유지보수성 향상 |

Pydantic은 **FastAPI를 FastAPI답게 사용하는 핵심 기능**입니다.

### 복습 문제 Pydantic 사용 시 얻는 장점은 무엇인가요?  
<details> <summary>정답 보기</summary>

자동 데이터 검증과 타입 체크를 통해 서버의 안정성을 높일 수 있다.
</details>

## 6. FastAPI 예시 및 연습 문제

아래 예시와 문제는 모두 **`main.py`에 이어서 추가**하는 형태입니다. (`app = FastAPI()`는 이미 있다고 가정)  
그래서 API 경로·함수 이름·클래스 이름이 서로 겹치지 않게 지어져 있습니다.

### 기초 예시

#### 예시 1. 회원가입 API (POST /signup)

##### 요구사항
- 사용자가 이름, 이메일, 나이를 보내면 서버는 “회원가입 완료” 메시지를 반환한다.

##### 요청(JSON)
```json
{
  "name": "김철수",
  "email": "test@example.com",
  "age": 25
}
```

##### 응답(JSON)
```json
{
  "message": "회원가입 완료!",
  "user": {
    "name": "김철수",
    "email": "test@example.com",
    "age": 25
  }
}
```

##### 예시 코드
```python
from pydantic import BaseModel

# Request Body 데이터 구조 정의 (Pydantic)
class SignupRequest(BaseModel):
    name: str                                       # 이름
    email: str                                      # 이메일
    age: int                                        # 나이 (정수)

@app.post("/signup")
def signup(data: SignupRequest):                    # data는 Pydantic으로 자동 검증된 객체
    return {
        "message": "회원가입 완료!",
        "user": data                                # Pydantic 객체도 그대로 넣으면 JSON으로 변환됨
    }
```

#### 예시 2. BMI 계산 API (POST /bmi)

##### 요구사항
- 키(cm)와 몸무게(kg)를 입력하면 BMI를 계산해줌  
  BMI = 몸무게(kg) / (키(m)^2)

##### 요청(JSON)
```json
{
  "height": 170,
  "weight": 65
}
```

##### 응답(JSON)
```json
{
  "height": 170,
  "weight": 65,
  "bmi": 22.49
}
```

##### 예시 코드
```python
from pydantic import BaseModel

class BMIRequest(BaseModel):
    height: float                                   # 키(cm)
    weight: float                                   # 몸무게(kg)

@app.post("/bmi")
def calculate_bmi(data: BMIRequest):
    height_m = data.height / 100                    # cm → m
    bmi = data.weight / (height_m * height_m)       # BMI = 몸무게 / 키(m)²

    return {
        "height": data.height,
        "weight": data.weight,
        "bmi": round(bmi, 2)                        # 소수점 둘째 자리까지 반올림
    }
```


#### 예시 3. 로그인 API (POST /login)

##### 요구사항
- id, password를 입력하면 성공 여부를 반환

##### 요청
```json
{
  "id": "admin",
  "password": "1234"
}
```

##### 응답
```json
{
  "login": true,
  "message": "로그인 성공"
}
```

##### 예시 코드
```python
from pydantic import BaseModel

class LoginRequest(BaseModel):
    id: str                                         # 아이디
    password: str                                   # 비밀번호

@app.post("/login")
def login(data: LoginRequest):
    # 실제 서비스에서는 DB 확인 필요
    if data.id == "admin" and data.password == "1234":  # 정해진 계정과 일치하는지 비교
        return {"login": True, "message": "로그인 성공"}
    else:
        return {"login": False, "message": "로그인 실패"}
```

#### 예시 4. 상품 등록 API (POST /product)

##### 요구사항
- 상품명, 가격, 재고를 입력하면 “등록 완료” 메시지 반환

##### 요청
```json
{
  "name": "사과",
  "price": 1200,
  "stock": 50
}
```

##### 응답
```json
{
  "message": "상품 등록 완료",
  "product": {
    "name": "사과",
    "price": 1200,
    "stock": 50
  }
}
```

##### 코드
```python
from pydantic import BaseModel

class Product(BaseModel):
    name: str                                       # 상품명
    price: int                                      # 가격
    stock: int                                      # 재고

@app.post("/product")
def add_product(data: Product):
    return {
        "message": "상품 등록 완료",
        "product": data                             # 받은 상품 정보를 그대로 응답에 포함
    }
```

#### 예시 5. 좌표 거리 계산 API (POST /distance)

##### 요구사항
- 두 좌표 (x1, y1), (x2, y2)를 입력하면 두 점 사이 거리 반환

거리 공식:  
distance = √((x2 - x1)² + (y2 - y1)²)

##### 요청
```json
{
  "x1": 0,
  "y1": 0,
  "x2": 3,
  "y2": 4
}
```

##### 응답
```json
{
  "distance": 5.0
}
```

##### 코드
```python
from pydantic import BaseModel
import math                                         # 제곱근(sqrt) 계산용

class PointDistance(BaseModel):
    x1: float                                       # 첫 번째 점
    y1: float
    x2: float                                       # 두 번째 점
    y2: float

@app.post("/distance")
def calculate_distance(data: PointDistance):
    dx = data.x2 - data.x1                          # x 좌표 차이
    dy = data.y2 - data.y1                          # y 좌표 차이
    dist = math.sqrt(dx*dx + dy*dy)                 # 피타고라스 정리
    return {"distance": dist}
```

### 연습문제 (쉬움)

모든 문제는 **Pydantic 모델로 Request Body를 받도록** 작성해 보세요. (정답 코드도 Pydantic을 사용합니다)

#### 문제 1. 인사 메시지 만들기 API

**요구사항**  
- POST `/greet`  
- Request Body(JSON):  
  - name: 문자열  
- 요청 예시: `{"name": "홍길동"}`  
- 응답 예시  
  ```json
  { "message": "안녕하세요, 홍길동님!" }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class GreetRequest(BaseModel):
    name: str                                       # 이름

@app.post("/greet")
def greet(request: GreetRequest):
    return {"message": f"안녕하세요, {request.name}님!"}  # f-string으로 이름 끼워 넣기
```

</details>

#### 문제 2. 두 숫자의 합과 평균 API

**요구사항**  
- POST `/math`  
- Request Body(JSON):  
  - a: 정수  
  - b: 정수  
- 요청 예시: `{"a": 10, "b": 20}`  
- 평균 = (a + b) / 2  
- 응답 예시  
  ```json
  { "sum": 30, "avg": 15.0 }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class MathRequest(BaseModel):
    a: int
    b: int

@app.post("/math")
def sum_and_avg(request: MathRequest):              # 함수 이름을 math로 지으면 math 모듈을 덮어쓰므로 다른 이름 사용
    s = request.a + request.b                       # 합
    return {"sum": s, "avg": s / 2}                 # 합과 평균
```

</details>



#### 문제 3. 나이를 받아 성인 여부 판별 API

**요구사항**  
- POST `/check-age`  
- Request Body(JSON):  
  - age: 정수  
- 요청 예시: `{"age": 20}`  
- **19세 이상**이면 성인(`true`), 아니면 `false`  
- 응답 예시  
  ```json
  { "is_adult": true }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class AgeRequest(BaseModel):
    age: int                                        # 나이

@app.post("/check-age")
def check_age(request: AgeRequest):
    return {"is_adult": request.age >= 19}          # 19세 이상이면 True
```

</details>

#### 문제 4. 상품 가격 계산 API

**요구사항**  
- POST `/product/total`  
- Request Body(JSON):  
  - name: 문자열  
  - price: 정수  
  - count: 정수 (수량)  
- 요청 예시: `{"name": "사과", "price": 1200, "count": 10}`  
- total = price × count  
- 응답 예시  
  ```json
  { "name": "사과", "total": 12000 }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class ProductRequest(BaseModel):
    name: str                                       # 상품명
    price: int                                      # 단가
    count: int                                      # 수량

@app.post("/product/total")                         # 예시 4의 /product와 겹치지 않는 경로
def product_total(req: ProductRequest):
    return {"name": req.name, "total": req.price * req.count}  # 총액 = 단가 × 수량
```

</details>

#### 문제 5. 문자열 분석 API

**요구사항**  
- POST `/analyze`  
- Request Body(JSON):  
  - text: 문자열  
- 요청 예시: `{"text": "hello fastapi 123"}`  
- length = text의 글자 수 (공백 포함)  
- 응답 예시  
  ```json
  { "length": 17 }
  ```

<details><summary>정답</summary>

```python
from pydantic import BaseModel

class AnalyzeRequest(BaseModel):
    text: str                                       # 분석할 문자열

@app.post("/analyze")
def analyze(request: AnalyzeRequest):
    return {"length": len(request.text)}            # len()으로 글자 수 계산
```

</details>

### 중급 예시

중급 예시는 실제 API 구조에 자주 등장하는 패턴을 포함합니다.  
각 코드에는 충분한 주석이 달려 있어, 문제를 풀기 위한 참고 자료 역할을 합니다.

#### "회원 가입 정보"를 받는 API (email은 선택값)
선택값 + 기본값 예시

```python
from pydantic import BaseModel

class SignupOptional(BaseModel):
    username: str                                   # 필수
    age: int                                        # 필수
    email: str | None = None                        # 없어도 OK (기본값 None)
    is_active: bool = True                          # 없으면 True

@app.post("/signup/optional")
def signup_optional(user: SignupOptional):
    return {
        "username": user.username,
        "age": user.age,
        "email": user.email,                        # 보내지 않았으면 None(null)
        "is_active": user.is_active                 # 보내지 않았으면 True
    }
```


#### 나이와 점수를 검증하는 API  
Pydantic 유효성 검증 (ge, le 사용)

- 나이(age)는 0 이상  
- 점수(score)는 0~100 사이

```python
from pydantic import BaseModel, Field

class ScoreRequest(BaseModel):
    age: int = Field(..., ge=0)                     # ... = 필수, ge=0 → age >= 0
    score: int = Field(..., ge=0, le=100)           # 0 <= score <= 100

@app.post("/validate-score")
def validate(req: ScoreRequest):                    # 범위를 벗어나면 함수 실행 전에 422 반환
    return {"status": "ok"}
```

#### 배송 정보(주소 + 사용자) 받기
중첩 모델(Nested Model)


```python
from pydantic import BaseModel

class DeliveryAddress(BaseModel):
    city: str                                       # 도시
    zipcode: str                                    # 우편번호

class Receiver(BaseModel):
    name: str                                       # 받는 사람 이름
    address: DeliveryAddress                        # 다른 모델을 포함 (중첩)

@app.post("/order")
def order(receiver: Receiver):
    return {
        "name": receiver.name,
        "city": receiver.address.city,              # 점(.)을 이어서 안쪽 값에 접근
        "zipcode": receiver.address.zipcode
    }
```

#### 여러 상품 가격 합산 API
리스트(List) 구조 받기

JSON 배열을 받아서 총합 계산

```python
from pydantic import BaseModel

class CartItem(BaseModel):
    name: str                                       # 상품명
    price: int                                      # 가격

class Cart(BaseModel):
    items: list[CartItem]                           # 여러 상품을 리스트로 받음

@app.post("/cart/total")
def cart_total(cart: Cart):
    total = sum(item.price for item in cart.items)  # 모든 상품 가격 합산
    return {"total": total}
```

#### 특정 user_id의 활동 기록을 저장하는 API
Path + Query + Body 조합


- Path: user_id  
- Query: detailed (True/False)  
- Body: 활동명, 지속시간

```python 
from pydantic import BaseModel

class Activity(BaseModel):
    name: str                                       # 활동명
    minutes: int                                    # 지속시간(분)

@app.post("/users/{user_id}/activity")
def add_activity(user_id: int, activity: Activity, detailed: bool = False):  # Path / Body / Query
    return {
        "user_id": user_id,
        "activity": activity.name,
        "minutes": activity.minutes,
        "detailed": detailed
    }
```

### 연습문제 (중간)

#### 문제 1. Optional 필드 포함한 회원 등록 API 만들기

##### 요구사항  
- POST `/register`  
- Request Body(JSON)  
  - username: 문자열  
  - age: 정수  
  - email: 문자열 (선택, 없으면 None)  
  - marketing: bool (선택, 기본값 False)  
- 응답: 입력된 모든 값을 그대로 JSON으로 반환

##### 추가 설명  
- email이 없어도 API가 정상 작동해야 함  
- marketing 값이 없으면 False 처리되어야 함

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel

class Register(BaseModel):
    username: str
    age: int
    email: str | None = None                        # 선택 (기본값 None)
    marketing: bool = False                         # 선택 (기본값 False)

@app.post("/register")
def register(user: Register):
    return user.model_dump()                        # model_dump() : Pydantic 객체 → dict로 변환해 반환
```

</details>

#### 문제 2. 숫자 검증이 포함된 API 만들기

##### 요구사항  
- POST `/score/check`  
- Request Body(JSON)  
  - math: 정수 (0~100)  
  - english: 정수 (0~100)  
- 요청 예시: `{"math": 90, "english": 80}`  
- 응답 예시  
  ```json
  { "total": 170 }
  ```

##### 조건  
Pydantic Field를 활용하여 0~100 범위를 벗어나면 오류(422)가 반환되도록 해야 함.

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel, Field

class Score(BaseModel):
    math: int = Field(..., ge=0, le=100)            # 0~100만 허용
    english: int = Field(..., ge=0, le=100)         # 0~100만 허용

@app.post("/score/check")
def check(score: Score):
    total = score.math + score.english              # 두 과목 합계
    return {"total": total}
```

</details>

#### 문제 3. 중첩 모델을 이용한 주문 생성 API

##### 요구사항  
- POST `/order/create`  
- Request Body(JSON)
  ```json
  {
    "customer": {"name": "홍길동", "phone": "010-1234-5678"},
    "address": {"city": "서울", "zipcode": "12345"}
  }
  ```
- 응답: 이름과 도시를 묶어 다음 JSON으로 응답
  ```json
  {
    "summary": "홍길동님, 서울로 배송됩니다."
  }
  ```

##### 조건  
- customer와 address 두 개의 모델을 만들 것  
- Order 모델을 만들어 두 모델을 포함시키기

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel

class Customer(BaseModel):
    name: str                                       # 고객 이름
    phone: str                                      # 전화번호

class Address(BaseModel):
    city: str                                       # 도시
    zipcode: str                                    # 우편번호

class Order(BaseModel):
    customer: Customer                              # 중첩 모델 1
    address: Address                                # 중첩 모델 2

@app.post("/order/create")
def create_order(order: Order):
    msg = f"{order.customer.name}님, {order.address.city}로 배송됩니다."  # 두 모델의 값을 조합
    return {"summary": msg}
```

</details>



#### 문제 4. 여러 상품을 받아 총 금액과 개수를 계산하는 API

##### 요구사항  
- POST `/items/summary`  
- Request Body(JSON)
  ```json
  {
    "items": [
      {"name": "사과", "price": 3000},
      {"name": "바나나", "price": 2000}
    ]
  }
  ```
- 응답
  ```json
  {
    "total_price": 5000,
    "count": 2
  }
  ```

##### 조건  
- list[Item] 형태로 모델 구성  
- sum()과 len()을 활용

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel

class Item(BaseModel):
    name: str                                       # 상품명
    price: int                                      # 가격

class ItemList(BaseModel):
    items: list[Item]                               # Item 여러 개

@app.post("/items/summary")
def items_summary(data: ItemList):
    total = sum(item.price for item in data.items)  # 가격 합계
    count = len(data.items)                         # 상품 개수
    return {"total_price": total, "count": count}
```

</details>

#### 문제 5. Path + Query + Body 조합 API 만들기

##### 요구사항  
- POST `/users/{user_id}/log`  
- Path Parameter  
  - user_id (int)  
- Query Parameter  
  - detail (bool, 기본값 False)  
- Request Body(JSON)  
  - action: 문자열  
  - duration: 정수  
- detail이 true면 `"detail": "상세 로그 저장됨"` 추가  
- detail이 false면 `"detail": "간단 로그 저장됨"` 추가

##### 요청 예시  
`POST /users/1/log?detail=true`
```json
{
  "action": "운동",
  "duration": 30
}
```

##### 응답 예시  
```json
{
  "user_id": 1,
  "action": "운동",
  "duration": 30,
  "detail": "상세 로그 저장됨"
}
```

<details><summary>정답 보기</summary>

```python
from pydantic import BaseModel

class Log(BaseModel):
    action: str                                     # 활동 이름
    duration: int                                   # 지속 시간

@app.post("/users/{user_id}/log")
def log_activity(user_id: int, log: Log, detail: bool = False):  # Path / Body / Query
    msg = "상세 로그 저장됨" if detail else "간단 로그 저장됨"  # detail 값에 따라 메시지 선택
    return {
        "user_id": user_id,
        "action": log.action,
        "duration": log.duration,
        "detail": msg
    }
```

</details>

### 고급 예시

#### 예시 1. 이미지 파일 업로드 + 폼 모델 함께 받기  
(파일과 폼 데이터를 함께 처리하는 고급 패턴)

#### ✔ 복습: 폼 데이터

5.3에서 본 것처럼, 파일을 함께 보낼 때는 Body를 JSON이 아니라 **폼 데이터(`multipart/form-data`)** 로 포장합니다.

이때 설명·카테고리 같은 글자 값도 폼 데이터의 한 칸으로 함께 전달됩니다.

```python
from fastapi import UploadFile, File, Form
from pydantic import BaseModel

class ImageMeta(BaseModel):                         # 이미지 정보를 묶어 관리할 모델
    description: str                                # 이미지 설명
    category: str                                   # 카테고리

@app.post("/upload")
async def upload_image(
    description: str = Form(...),                   # 폼 필드 description
    category: str = Form(...),                      # 폼 필드 category
    file: UploadFile = File(...)                    # 업로드 파일
):
    meta = ImageMeta(description=description, category=category)  # 받은 폼 값을 Pydantic 모델로 묶기

    return {
        "description": meta.description,
        "category": meta.category,
        "filename": file.filename                   # 업로드된 파일 이름
    }
```

✔ 특징
- 관련된 폼 필드를 받은 뒤 Pydantic 모델로 묶어 검증하거나 관리할 수 있음
- `Form()`을 사용해 JSON이 아닌 **폼 데이터에서 값 추출**
- 파일과 함께 `multipart/form-data` 요청 처리 가능

##### 참고: curl 테스트 명령어

```bash
curl -X POST http://localhost:8000/upload \
  -F 'description=귀여운 강아지' \
  -F 'category=animal' \
  -F 'file=@image/dog.jpg'
```



##### 명령어 해설

- `-F 'description=귀여운 강아지'`  
  → 텍스트 폼 필드 전송  

- `-F 'category=animal'`  
  → 텍스트 폼 필드 전송  

- `-F 'file=@image/dog.jpg'`  
  → 파일 업로드 (`@`는 파일 경로 의미)




#### ✔ 정리

이 요청은 웹페이지에서:

1. 설명 입력  
2. 카테고리 입력  
3. 파일 첨부  
4. 제출 버튼 클릭  

하는 상황을 `curl`로 그대로 재현한 것이라고 보면 됩니다.

#### 예시 2. 여러 파일 업로드 + 파일 개수 검증

```python
from fastapi import UploadFile, File, HTTPException

@app.post("/multi-upload")
async def multi_upload(files: list[UploadFile] = File(...)):  # 같은 이름(files)으로 여러 파일 받기
    if len(files) > 5:                              # 6개 이상이면 거절
        raise HTTPException(status_code=400, detail="파일은 최대 5개까지만 가능합니다.")  # 400 + 에러 메시지 응답

    names = [f.filename for f in files]             # 파일 이름만 모으기
    return {"uploaded_files": names}
```

✔ 특징
- 리스트 형태의 파일 받기
- 조건 불만족 시 HTTPException 발생
- 실제 서비스에서 매우 자주 쓰임

#### 예시 3. Pydantic 모델 + 검증 + 리스트 + 중첩 종합 패턴

```python
from pydantic import BaseModel, Field

class OrderItem(BaseModel):
    name: str
    price: int = Field(..., ge=0)                   # 가격은 0 이상
    count: int = Field(..., ge=1)                   # 수량은 1 이상

class CartOrder(BaseModel):
    user: str                                       # 주문자
    items: list[OrderItem]                          # OrderItem 여러 개 (중첩 + 리스트)

@app.post("/orders/calc")
def calc_order(order: CartOrder):
    total = sum(item.price * item.count for item in order.items)  # 상품별 (가격 × 수량)의 합
    return {
        "user": order.user,
        "items": len(order.items),                  # 상품 종류 수
        "total_price": total
    }
```

✔ 특징
- 중첩 모델(CartOrder → OrderItem)
- 검증 포함(price >= 0, count >= 1)
- 리스트 합산 로직
- 실제 쇼핑몰 API 구조

## 7. 보안 설정 - CORS(Cross-Origin Resource Sharing)

### 7.1 CORS란?

웹 브라우저는 보안을 위해,
**현재 보고 있는 웹페이지와 다른 주소의 서버로 보낸 요청의 결과를, 그 웹페이지의 JavaScript가 마음대로 읽지 못하도록** 기본적으로 막습니다.  
이 기본 규칙을 **동일 출처 정책(Same-Origin Policy)** 이라고 합니다.

예를 들어,

- 프론트엔드: `http://localhost:3000`
- 백엔드 API: `http://localhost:8000`

처럼 주소가 다르면, 브라우저는 이 요청을 바로 허용하지 않습니다.

이 제한을 **서버가 허용한 출처에 한해서 풀어주는 방식**이 **CORS(Cross-Origin Resource Sharing)** 입니다.

쉽게 말하면:

> “지금 보고 있는 웹페이지와 다른 서버에 요청하려고 하네?  
> 서버가 허용한 경우에만 보내줄게.”

이런 제한이 없으면 악성 사이트가 사용자를 “중간 발판”처럼 써서 개인 데이터를 빼낼 수 있습니다.
- 예시 :
    1. 사용자가 은행에 로그인한 상태에서 악성 사이트에 들어감.
    2. 그 사이트 안의 자바스크립트가 몰래 은행 서버에 계좌 조회·개인정보 조회 요청을 보내고, 그 응답을 읽어 가려고 함

브라우저가 이런 교차 출처 응답을 다 넘겨주면, 사용자는 그냥 웹페이지 하나 봤을 뿐인데  
 자기 계좌 정보가 악성 사이트로 넘어갈 수 있음

### 7.2 Origin(출처)이란?

브라우저는 아래 3가지가 모두 같을 때만 **같은 출처(Same Origin)** 라고 판단합니다.

- 프로토콜: `http` / `https`
- 도메인: `localhost` / `example.com`
- 포트: `3000` / `8000`

이 중 하나라도 다르면 **다른 출처(Cross Origin)** 입니다.

예:

- `http://localhost:8000`
- `http://localhost:3000`

위 두 주소는 **포트가 다르므로 다른 출처**입니다.

즉, 같은 컴퓨터에서 실행 중이어도  
포트가 다르면 브라우저는 다른 사이트로 봅니다.

### 7.3 언제 CORS 문제가 생기나요?

CORS 문제는 주로 다음 상황에서 발생합니다.

- **브라우저에서 실행 중인 프론트엔드(JavaScript)가**
- **다른 출처의 API 서버에 요청할 때**

예:

```text
프론트엔드: http://localhost:3000
백엔드 API: http://localhost:8000
```

이 경우 브라우저는 보안상 요청을 검사하고,  
서버가 허용하지 않으면 차단합니다.

그래서 개발 중 아래와 같은 에러를 자주 보게 됩니다.

```text
Blocked by CORS policy
```

### 7.4 언제 CORS 에러가 안 나나요?

중요한 점은, **CORS는 브라우저의 보안 정책**이라는 것입니다.

그래서 아래처럼 **브라우저 밖에서 보내는 요청**에는 보통 CORS가 적용되지 않습니다.

- `requests`
- `curl`
- Postman
- 서버 간 통신

즉, 같은 API라도

- 브라우저에서 호출하면 CORS 에러가 날 수 있고
- `curl`이나 Postman에서는 잘 동작할 수 있습니다.

### 7.5 왜 FastAPI에서 CORS를 설정하나요?

브라우저가 다른 출처의 요청을 보내려고 할 때,  
서버가 “이 출처는 허용합니다”라고 응답해줘야  
브라우저가 요청을 허용합니다.

즉,

- **요청을 보내는 쪽**: 프론트엔드 JavaScript
- **요청을 검사하는 쪽**: 브라우저
- **허용 여부를 알려주는 쪽**: 백엔드(FastAPI)

그래서 CORS 설정은 **백엔드에서** 합니다.

### 7.6 FastAPI에서 CORS 설정하기

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],   # 모든 출처 허용
    allow_methods=["*"],   # 모든 HTTP 메서드 허용
    allow_headers=["*"],   # 모든 헤더 허용
)
```

이 설정은 브라우저에게 다음과 같이 말하는 것과 같습니다.

> “어떤 출처에서 온 요청이든 허용하겠습니다.”


### 7.7 `allow_origins=["*"]`는 왜 조심해야 하나요?

`allow_origins=["*"]`는 개발할 때는 편리하지만,  
실서비스에서는 너무 넓게 허용하는 설정입니다.

실제 서비스에서는 보통 허용할 프론트엔드 주소만 명시합니다.

```python
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:3000",
        "https://my-frontend.com"
    ],
    allow_methods=["*"],
    allow_headers=["*"],
)
```

이렇게 하면 지정한 출처에서 오는 요청만 브라우저가 허용합니다.

### 7.8 핵심 정리

- CORS는 **브라우저의 보안 정책**이다.
- 브라우저는 **다른 출처의 요청을 기본적으로 제한**한다.
- 이때 서버가 허용 응답을 주면 요청이 가능해진다.
- 그래서 CORS 설정은 **백엔드에서** 한다.
- 개발 중에는 전체 허용으로 시작할 수 있지만,
  배포 시에는 **허용할 출처만 구체적으로 지정하는 것이 안전**하다.



### 7.9 한 줄 요약

> CORS는  
> **브라우저가 다른 출처의 요청을 제한할 때, 서버가 허용 여부를 알려주는 방식**이다.

### 복습 문제. CORS 설정의 목적을 한 문장으로 쓰시오.
<details> <summary>정답 보기</summary>

브라우저에서 실행되는 다른 출처의 웹페이지(프론트엔드)가 이 API를 호출할 수 있도록, 허용할 출처를 서버가 브라우저에게 알려주기 위해 설정한다.
</details>

## 8. 실습 예제: 이미지 분류 모델 서빙

### 8.1 준비
지금까지 배운 내용을 모아, **사전 학습된 이미지 분류 모델(ResNet18)** 을 API로 서빙해 봅니다.

실습 흐름:
1. 서버 코드 파일(`model_server.py`) 작성
2. 터미널에서 서버 실행
3. Swagger UI, curl, Python 코드로 요청 보내기

필요한 패키지 설치:
```bash
pip install "fastapi[standard]" torch torchvision pillow requests
```

### 8.2 서빙 코드 작성 (`model_server.py`)

서빙 코드를 짤 때 기억할 4가지:

| 포인트 | 이유 |
|---|---|
| 모델은 서버 시작 시 **한 번만** 로드 | 요청마다 로드하면 매번 수 초씩 걸림 |
| 전처리는 **학습 때와 똑같이** | `weights.transforms()`가 학습 때 쓴 전처리를 그대로 제공 |
| 결과는 **사람이 읽을 수 있게** | 클래스 번호(259) 대신 이름(Pomeranian)과 확률로 응답 |
| 잘못된 입력은 **400으로 거절** | 이미지가 아닌 파일이 와도 서버가 죽지 않도록 |

아래 셀을 실행하면 `model_server.py` 파일이 만들어집니다.

In [ ]:
%%writefile model_server.py
import io                                                    # 업로드된 바이트를 파일처럼 다루기 위한 모듈

import torch                                                 # 추론 실행용 PyTorch
from fastapi import FastAPI, File, HTTPException, UploadFile # FastAPI 핵심 클래스와 파일 업로드 도구
from PIL import Image, UnidentifiedImageError                # 이미지 열기 / 이미지가 아닐 때 나는 에러
from pydantic import BaseModel                               # 응답 형식을 정의할 Pydantic 모델
from torchvision.models import ResNet18_Weights, resnet18    # 사전 학습된 ResNet18과 가중치 정보

app = FastAPI(title="ResNet18 이미지 분류 API")              # Swagger 문서에 표시될 제목

# ---------- 1) 모델 준비: 서버가 시작될 때 딱 한 번 실행 ----------
weights = ResNet18_Weights.DEFAULT                           # ImageNet으로 학습된 기본 가중치
model = resnet18(weights=weights)                            # 모델 구조 생성 + 가중치 로드
model.eval()                                                 # 추론 모드 (Dropout/BatchNorm 동작 고정)
preprocess = weights.transforms()                            # 학습 때 쓰인 전처리 (Resize → Crop → Normalize)
categories = weights.meta["categories"]                      # 클래스 번호 → 이름 목록 (1000개)

# ---------- 2) 응답 형식 정의 ----------
class Prediction(BaseModel):                                 # 예측 1건
    label: str                                               # 클래스 이름 (예: "Pomeranian")
    confidence: float                                        # 확률 (0~1)

class PredictResponse(BaseModel):                            # /predict 응답 전체
    filename: str                                            # 업로드한 파일 이름
    top3: list[Prediction]                                   # 확률 높은 순 3개

# ---------- 3) 엔드포인트 ----------
@app.get("/health")                                          # 서버가 살아 있는지 확인용
def health():
    return {"status": "ok"}                                  # 응답이 오면 정상

@app.post("/predict", response_model=PredictResponse)        # 응답이 PredictResponse 형식인지 검사·문서화
def predict(file: UploadFile = File(...)):                   # async 없이 def → 추론 중에도 다른 요청 처리 가능
    image_bytes = file.file.read()                           # 업로드 파일 내용을 바이트로 읽기
    try:
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")  # 흑백·투명 PNG도 RGB 3채널로 통일
    except UnidentifiedImageError:                           # 이미지가 아닌 파일이 올라온 경우
        raise HTTPException(status_code=400, detail="이미지 파일만 업로드할 수 있습니다.")  # 500 대신 400으로 거절

    x = preprocess(image).unsqueeze(0)                       # (3, 224, 224) → 배치 차원 추가 (1, 3, 224, 224)
    with torch.inference_mode():                             # 기울기 계산 끄기 → 메모리 절약, 속도 향상
        probs = model(x).softmax(dim=1)[0]                   # 출력(로짓) → 확률, 첫 번째(유일한) 이미지 선택
    top_probs, top_ids = probs.topk(3)                       # 확률 상위 3개의 값과 클래스 번호

    return {                                                 # dict로 반환해도 response_model에 맞춰 검증됨
        "filename": file.filename,
        "top3": [
            {"label": categories[i], "confidence": round(p, 4)}  # 번호를 이름으로 바꾸고 소수 넷째 자리까지
            for p, i in zip(top_probs.tolist(), top_ids.tolist())
        ],
    }

#### 코드 설명
- **모델 준비 (파일 위쪽)**: 서버가 켜질 때 한 번만 실행되고, 이후 모든 요청이 같은 모델을 함께 사용
- **`.convert("RGB")`**: 흑백 사진, 투명 배경 PNG처럼 채널 수가 다른 이미지도 3채널로 맞춰 모델에 넣을 수 있게 함
- **`HTTPException(400)`**: 이미지가 아닌 파일이 오면 500(서버 오류) 대신 400(잘못된 요청)과 이유를 응답
- **`response_model=PredictResponse`**: 응답 형식을 Pydantic으로 정의 → Swagger 문서에 응답 구조가 표시됨
- **`/health`**: 서버가 정상 동작 중인지 간단히 확인하는 엔드포인트

#### 왜 `async def`가 아니라 `def`인가?
3.8에서 본 것처럼, 비동기가 아껴주는 것은 **기다리는 시간**입니다.  
모델 추론은 기다리는 작업이 아니라 CPU·GPU가 **계속 계산하는** 작업이라 상황이 다릅니다.

| 선언 | 추론이 도는 동안 다른 요청은? |
|---|---|
| `async def` | 추론이 끝날 때까지 **서버 전체가 멈춤** (`/health`조차 응답 못 함) |
| `def` | FastAPI가 별도 스레드에서 실행 → **다른 요청은 계속 처리** |

> 규칙: **계산하는 작업(모델 추론) → `def`**, **기다리는 작업(DB·외부 API 호출) → `async def`**

### 8.3 서버 실행
**터미널**에서 `model_server.py`가 있는 폴더로 이동한 뒤 실행합니다.  
앞에서 실행한 `main.py` 서버가 켜져 있다면 `Ctrl + C`로 먼저 종료합니다. (둘 다 8000번 포트를 사용)

```bash
uvicorn model_server:app --reload
```

처음 실행할 때는 모델 가중치(약 45MB)를 내려받기 때문에 조금 기다려야 합니다.  
터미널에 `Application startup complete.`가 보이면 준비 완료입니다.

### 8.4 테스트

#### 방법 1) Swagger UI
- 접속 주소: [http://127.0.0.1:8000/docs](http://127.0.0.1:8000/docs)
- `POST /predict` → `Try it out` → 이미지 파일 선택 → `Execute`
- 아래쪽 **Responses**에서 예측 결과 확인

#### 방법 2) curl
```bash
curl -X POST http://127.0.0.1:8000/predict -F "file=@image/dog.jpg"
```

#### 방법 3) Python 코드 (`requests`)
실제 서비스에서는 웹·앱·다른 서버가 이런 방식으로 API를 호출합니다.

In [ ]:
import requests                                              # HTTP 요청을 보내는 라이브러리

URL = "http://127.0.0.1:8000"                                # 모델 서버 주소

res = requests.get(f"{URL}/health")                          # 서버 상태 확인
print(res.json())                                            # {'status': 'ok'} 이면 정상

In [ ]:
with open("image/dog.jpg", "rb") as f:                       # 이미지를 바이너리(rb) 모드로 열기
    res = requests.post(f"{URL}/predict", files={"file": f}) # "file" = 엔드포인트의 매개변수 이름

print(res.status_code)                                       # 200 이면 성공
result = res.json()                                          # 응답 JSON → dict
for p in result["top3"]:                                     # 확률 높은 순 3개 출력
    print(f'{p["label"]:<20} {p["confidence"]:.1%}')         # 이름(왼쪽 정렬 20칸)과 확률(%)

이미지가 아닌 파일을 보내면, 서버가 죽지 않고 **400**과 이유를 돌려줍니다.

In [ ]:
fake = ("memo.txt", b"hello", "text/plain")                  # (파일 이름, 내용, 파일 형식)으로 만든 텍스트 파일
res = requests.post(f"{URL}/predict", files={"file": fake})  # 이미지 대신 텍스트 파일 업로드
print(res.status_code, res.json())                           # 400과 에러 메시지

### 복습 문제

**문제 1.** FastAPI에서 업로드된 이미지 파일을 받으려면 어떤 클래스를 사용하나요?

<details> <summary>정답 보기</summary>

`UploadFile` 클래스와 `File(...)` 선언을 사용한다.  
예: `def predict(file: UploadFile = File(...)):`
</details>

**문제 2.** 모델 추론 엔드포인트를 `async def`가 아니라 `def`로 선언하는 이유는 무엇인가요?

<details> <summary>정답 보기</summary>

모델 추론은 기다리는 작업이 아니라 계산하는 작업이어서, `async def` 안에서 실행하면 추론이 끝날 때까지 서버 전체가 다른 요청을 처리하지 못한다.  
`def`로 선언하면 FastAPI가 별도 스레드에서 실행하므로 그동안에도 다른 요청을 처리할 수 있다.
</details>

**문제 3.** 모델을 요청마다 불러오지 않고 서버 시작 시 한 번만 불러오는 이유는 무엇인가요?

<details> <summary>정답 보기</summary>

모델 로드는 수 초가 걸리는 작업이라, 요청마다 반복하면 모든 응답이 그만큼 느려진다. 한 번 불러온 모델을 모든 요청이 함께 사용한다.
</details>

## 9. 배포 및 운영

### 9.1 클라우드 서버에 배포하기 (Amazon EC2)

AI 모델이나 웹 서비스를 실제 사용자에게 제공하려면, **클라우드 서버**에 배포해야 합니다.

대표적인 클라우드 서비스:
- **Google Cloud Platform (GCP)** → 무료 크레딧 제공
- **Amazon AWS (EC2 프리티어)** → 가장 대중적으로 많이 쓰이는 클라우드 서비스

#### EC2에서 FastAPI 실행 순서
1. EC2 인스턴스 생성 (Ubuntu 권장)
2. SSH로 서버 접속 (내 PC 터미널에서)
   ```bash
   ssh -i 내키.pem ubuntu@서버IP
   ```
3. 가상환경을 만들고 FastAPI 설치 (서버에서)
   ```bash
   sudo apt update && sudo apt install -y python3-venv   # 가상환경 도구 설치
   python3 -m venv .venv                                  # .venv 폴더에 가상환경 생성
   source .venv/bin/activate                              # 가상환경 활성화
   pip install "fastapi[standard]"                        # FastAPI + uvicorn 설치
   ```
4. FastAPI 코드(`main.py`) 업로드 (내 PC 터미널에서)
   ```bash
   scp -i 내키.pem main.py ubuntu@서버IP:~/
   ```
5. 서버 실행 (서버에서)
   ```bash
   nohup uvicorn main:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &
   ```
   - `--host 0.0.0.0`: 외부에서도 접속 가능하도록 설정
   - `--port 8000`: 서비스 포트 지정
   - `nohup ... &`: SSH 접속을 끊어도 서버가 계속 실행되도록 백그라운드에서 실행
   - `> server.log 2>&1`: 서버 로그를 `server.log` 파일에 저장 (`tail -f server.log`로 확인)
   - 개발용 옵션인 `--reload`는 배포할 때 빼고 실행합니다.

6. 보안 그룹(Security Group)에서 **포트 8000 열기** (인바운드 규칙 추가)  
   → 외부 사용자가 `http://서버IP:8000` 으로 접속 가능
7. 브라우저에서 `http://서버IP:8000/docs` 접속 → Swagger UI 확인

서버를 끌 때:
```bash
pkill -f uvicorn
```

### 9.2 Google Cloud(Compute Engine) 배포

흐름은 EC2와 같습니다. 다른 부분만 정리하면:

1. Compute Engine → **VM 인스턴스 생성** (Ubuntu 이미지 선택)
2. 콘솔의 **SSH 버튼**으로 브라우저에서 바로 접속 (키 파일 불필요)
3. 9.1의 3~5단계와 똑같이 설치·실행 (코드 파일은 SSH 창의 **파일 업로드** 메뉴로 올릴 수 있음)
4. **VPC 네트워크 → 방화벽 규칙**에서 `tcp:8000` 인바운드 허용
5. 브라우저에서 `http://외부IP:8000/docs` 접속

### 9.3 운영 고려사항

서비스를 단순히 실행하는 것만으로는 충분하지 않습니다.  
**운영 단계에서는 안정성과 보안**이 중요합니다.  

- **로그 모니터링**  
  → 에러, 요청 기록을 확인하여 문제 조기 발견  

- **서버 리소스 체크**  
  → CPU, 메모리, 디스크 사용량 모니터링 (예: htop, CloudWatch)  

- **Rate Limiting (요청 제한)**  
  → 특정 사용자가 과도하게 요청을 보내는 것을 방지 (DDOS 완화)  

- **보안 패치 및 업데이트**  
  → OS, Python 패키지를 주기적으로 업데이트하여 취약점 제거  

### 9.4 클라우드 배포 실습

본인이 생각하는 간단한 서비스를 FastAPI로 만들어 클라우드에 배포해 봅시다.

- 6장의 예시 API를 조합해서 만들어도 좋습니다.
- 배포 후 다른 사람의 PC(또는 스마트폰)에서 `http://서버IP:8000/docs`로 접속되는지 확인합니다.

### 복습 문제
EC2에 FastAPI 앱을 배포할 때, 외부에서 접근하려면 무엇을 설정해야 하나요?

<details> <summary>정답 보기</summary>

- 서버를 `--host 0.0.0.0`으로 실행해 외부 접속을 받도록 한다.
- Security Group에서 FastAPI가 사용하는 포트(예: 8000)를 개방한다.
</details>